In [ ]:
"""
今天主要学的是百分位数在阈值中的作用，Volatility Regime Detection Using Percentile Thresholds
核心思想为：将当前的波动比率和历史的波动比率进行比对

一般步骤为：
1.计算历史波动率
2.确定百分位数阈值
3.状态分类：（1）：低波动：低于较低百分位数阈值
        （2）：正常波动：位于两个百分位数阈值之间
        （3）：高波动：高于较高的百分位数阈值

"""

"""# 【任务 13】识别波动率异常日期
# 要求：
# 1. 计算 EWMA 波动率
# 2. 计算 EWMA 波动率的历史 95 分位数（用 expanding）
# 3. 找出当前波动率 > 95 分位数的日期
# --- 你的代码 ---

# ewma_vol = rm.ewma_volatility()
# threshold = ewma_vol.expanding().quantile(0.95)
# high_vol_dates = ewma_vol > threshold
# 
# # 统计每只股票的高波动天数
# high_vol_dates.sum()

# 【任务 14】生成风险摘要报告
# 要求：创建一个函数，输入 RiskMetrics 对象，输出包含以下内容的字典：
# - 最新波动率（所有资产）
# - 最新夏普比率
# - 当前回撤
# - 是否处于高波动状态
# - 过去 5 天的 VaR 突破次数
# --- 你的代码 ---

def generate_risk_report(rm: RiskMetrics) -> dict:
    report = {}
    
    # 最新值：well，因为这只是代码逻辑演示，因此，在真正使用的时候，不应该使用简单的rolling，而是应该使用ewm
    report['latest_volatility'] = rm.rolling_volatility(20).iloc[-1].to_dict()
    report['latest_sharpe'] = rm.rolling_sharpe(20).iloc[-1].to_dict()
    report['current_drawdown'] = rm.current_drawdown().iloc[-1].to_dict()
    
    # 高波动判断
    ewma_vol = rm.ewma_volatility()
    threshold = ewma_vol.expanding().quantile(0.95)
    report['high_volatility'] = (ewma_vol.iloc[-1] > threshold.iloc[-1]).to_dict()#这段代码对应的value是一个字典

    # VaR 突破：表示当日实际亏损多于VaR估计的最大亏损，VaR_95;如果是负值，则表示可以接受的最大亏损
    var_95 = rm.historical_var(confidence=0.95, window=252)
    var_breach = (rm.returns < var_95).tail(5).sum().to_dict()
    report['var_breach_last_5d'] = var_breach
    
    return report

# 测试
# report = generate_risk_report(rm)
# report

"""
